# 02 — Data Cleaning and Merging

**Goal:** Turn the raw wildfire records and raw NASA weather data into one clean table,
where each row is a fire with the weather on the day it was discovered.

Steps:
1. Load and clean the wildfire data (dates, names, impossible values, new variables)
2. Load and clean the weather data (missing-value codes, unit conversions)
3. Match each fire to the weather at its nearest NASA grid point on its discovery date
4. Save the result to `data/processed/`

At every step we print how many rows we have, so no data disappears silently.

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../src")  # lets us import our own helper functions from src/
from data_cleaning import (
    LARGE_FIRE_ACRES, CA_LAT_MIN, CA_LAT_MAX, CA_LON_MIN, CA_LON_MAX,
    load_california_fires, month_to_season, add_grid_cell_columns, clean_weather,
)

## Part 1: Wildfire data

### 1.1 Load
`load_california_fires()` lives in `src/data_cleaning.py`. It runs the same SQL query as Notebook 01,
keeping only the columns we need.

In [2]:
fires = load_california_fires("../data/raw/FPA_FOD_20221014.sqlite")
print("Rows loaded:", len(fires))

Rows loaded: 251881


### 1.2 Standardize column names
Lowercase names with underscores are easier to type and read.

In [3]:
fires = fires.rename(columns={
    "FOD_ID": "fire_id",
    "FIRE_NAME": "fire_name",
    "FIRE_YEAR": "fire_year",
    "DISCOVERY_DATE": "discovery_date",
    "NWCG_CAUSE_CLASSIFICATION": "cause_class",
    "NWCG_GENERAL_CAUSE": "cause",
    "FIRE_SIZE": "fire_size_acres",
    "FIRE_SIZE_CLASS": "fire_size_class",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "OWNER_DESCR": "land_owner",
    "STATE": "state",
    "FIPS_NAME": "county",
})
fires.columns.tolist()

['fire_id',
 'fire_name',
 'fire_year',
 'discovery_date',
 'cause_class',
 'cause',
 'fire_size_acres',
 'fire_size_class',
 'latitude',
 'longitude',
 'land_owner',
 'state',
 'county']

### 1.3 Convert dates
Notebook 01 showed dates are stored as text like `2/2/2005`. `pd.to_datetime` turns them into real dates.
`errors="coerce"` means any date that can't be read becomes missing (`NaT`) instead of crashing, so we can count them.

In [4]:
fires["discovery_date"] = pd.to_datetime(fires["discovery_date"], format="%m/%d/%Y", errors="coerce")
print("Dates that could not be read:", fires["discovery_date"].isna().sum())
print("Date range:", fires["discovery_date"].min().date(), "to", fires["discovery_date"].max().date())

Dates that could not be read: 0
Date range: 1992-01-01 to 2020-12-31


Check: does the year in the date always match the `fire_year` column?

In [5]:
year_mismatch = (fires["discovery_date"].dt.year != fires["fire_year"]).sum()
print("Rows where the date's year differs from fire_year:", year_mismatch)

Rows where the date's year differs from fire_year: 0


### 1.4 Remove duplicates
Notebook 01 found none, but checking again is cheap and makes the pipeline safe if the data changes.

In [6]:
rows_before = len(fires)
fires = fires.drop_duplicates(subset="fire_id")
print(f"Rows before: {rows_before:,}   after: {len(fires):,}   removed: {rows_before - len(fires):,}")

Rows before: 251,881   after: 251,881   removed: 0


### 1.5 Check for impossible values
- Fire size must be greater than 0 acres.
- The location must be inside (roughly) California.

In [7]:
print("Fires with size <= 0 acres:", (fires["fire_size_acres"] <= 0).sum())

inside_california = (
    fires["latitude"].between(CA_LAT_MIN, CA_LAT_MAX)
    & fires["longitude"].between(CA_LON_MIN, CA_LON_MAX)
)
print("Fires outside the California bounding box:", (~inside_california).sum())
fires.loc[~inside_california, ["fire_id", "fire_name", "discovery_date", "latitude", "longitude", "county"]]

Fires with size <= 0 acres: 0
Fires outside the California bounding box: 1


,fire_id,fire_name,discovery_date,latitude,longitude,county
235712,400502550,MIDDLE FORK,2019-06-14,42.4175,-123.568056,Siskiyou County


The record(s) outside the box are listed above. We remove them, because a location outside California
is either a data-entry error or a fire we can't reliably place. This is a very small number of rows.

In [8]:
rows_before = len(fires)
fires = fires[inside_california & (fires["fire_size_acres"] > 0)]
print(f"Rows before: {rows_before:,}   after: {len(fires):,}   removed: {rows_before - len(fires):,}")

Rows before: 251,881   after: 251,880   removed: 1


### 1.6 Handle missing values
From Notebook 01: `county` is missing for about 38% of fires and `fire_name` for about 19%.

**Decision: keep these rows.** Dropping them would throw away over a third of the data, and we don't need county
to match weather (we use latitude/longitude). We just label the missing values clearly.

In [9]:
fires["county"] = fires["county"].fillna("Unknown").str.replace(" County", "", regex=False)
fires["fire_name"] = fires["fire_name"].fillna("UNNAMED").str.strip().str.upper()
fires[["county", "fire_name"]].isna().sum()

county       0
fire_name    0
dtype: int64

### 1.7 Create new variables
- `fire_month`, `fire_season` — from the discovery date
- `log_fire_size` — `log(1 + acres)`. Fire size is extremely skewed (median 0.2 acres, max ~589,000),
  so on a normal scale almost every point is squashed near zero. The log spreads the values out so
  patterns are visible. We use `log1p` (log of 1 + x) so a size of 0 would still work.
- `is_large_fire` — `True` if the fire burned at least 100 acres (NWCG classes D–G).

In [10]:
fires["fire_month"] = fires["discovery_date"].dt.month
fires["fire_season"] = fires["fire_month"].apply(month_to_season)
fires["log_fire_size"] = np.log1p(fires["fire_size_acres"])
fires["is_large_fire"] = fires["fire_size_acres"] >= LARGE_FIRE_ACRES

print("Share of fires that are large (>= 100 acres):", round(fires["is_large_fire"].mean() * 100, 2), "%")
fires["fire_season"].value_counts()

Share of fires that are large (>= 100 acres): 2.15 %


fire_season
Summer    135056
Fall       60596
Spring     42001
Winter     14227
Name: count, dtype: int64

## Part 2: Weather data

### 2.1 Load
`data/raw/nasa_power_daily_ca.csv` was created by `src/download_weather.py`. It has one row per
grid point per day from Dec 1991 to Dec 2020, with NASA's original units:

| Column | Meaning | Units |
|---|---|---|
| `T2M_MAX` | Maximum air temperature at 2 m | °C |
| `RH2M` | Relative humidity at 2 m (daily average) | % |
| `WS2M` | Wind speed at 2 m (daily average) | m/s |
| `PRECTOTCORR` | Precipitation | mm/day |

In [11]:
weather_raw = pd.read_csv("../data/raw/nasa_power_daily_ca.csv")
print("Weather rows:", f"{len(weather_raw):,}")
print("Grid cells:", weather_raw[["cell_lat", "cell_lon"]].drop_duplicates().shape[0])
weather_raw.head()

Weather rows: 1,869,824
Grid cells: 176


,cell_lat,cell_lon,date,T2M_MAX,RH2M,WS2M,PRECTOTCORR
0,32.5,-117.5,19911201,15.86,40.83,3.69,0.00
1,32.5,-117.5,19911202,15.46,53.94,1.89,0.00
2,32.5,-117.5,19911203,18.19,55.17,3.97,0.01
3,32.5,-117.5,19911204,18.51,61.15,2.60,0.28
4,32.5,-117.5,19911205,16.90,84.63,1.85,0.17


### 2.2 Missing-value codes
NASA POWER uses `-999` to mean "no data". If we left it in, a temperature of -999 °C would wreck any average.

In [12]:
(weather_raw == -999).sum()

cell_lat       0
cell_lon       0
date           0
T2M_MAX        0
RH2M           0
WS2M           0
PRECTOTCORR    0
dtype: int64

### 2.3 Clean and convert units
`clean_weather()` (in `src/data_cleaning.py`) replaces `-999` with missing, converts the date,
converts units (°C → °F, m/s → mph), and adds `precip_prev_7day_mm`: the total rain in the **7 days before**
each date. One day's rain says little about how dry vegetation is; a week gives a better picture.

In [13]:
weather = clean_weather(weather_raw)
weather.describe().round(2)

,cell_lat,cell_lon,date,temp_max_f,relative_humidity,wind_speed_mph,precip_mm,precip_prev_7day_mm
count,1869824.00,1869824.00,1869824,1869824.00,1869824.00,1869824.00,1869824.00,1868592.00
mean,37.22,-119.62,2006-06-16 12:00:00,70.12,56.02,4.52,1.45,10.13
min,32.50,-124.38,1991-12-01 00:00:00,5.25,3.49,0.00,0.00,0.00
25%,35.00,-121.88,1999-03-09 18:00:00,56.93,34.65,2.48,0.00,0.04
50%,37.00,-120.00,2006-06-16 12:00:00,68.23,56.50,4.00,0.01,0.92
75%,39.50,-117.50,2013-09-23 06:00:00,84.09,77.71,5.93,0.28,8.92
max,42.50,-114.38,2020-12-31 00:00:00,124.72,100.00,30.11,192.18,470.24
std,2.79,2.70,NaN,18.51,24.35,3.05,5.11,22.45


Sanity check: are the values physically reasonable? Humidity must be 0–100%, and temperatures should be within
California's historic range (about −45 °F to 134 °F).

In [14]:
print("Humidity outside 0-100%:", ((weather["relative_humidity"] < 0) | (weather["relative_humidity"] > 100)).sum())
print("Max temp range (F):", round(weather["temp_max_f"].min(), 1), "to", round(weather["temp_max_f"].max(), 1))
print("Negative precipitation:", (weather["precip_mm"] < 0).sum())

Humidity outside 0-100%: 0
Max temp range (F): 5.3 to 124.7
Negative precipitation: 0


## Part 3: Combine fires with weather

**Method (grid-cell matching):**
NASA POWER's weather comes from a model grid with a point every 0.5° of latitude and 0.625° of longitude
(roughly 55 × 55 km in California). For each fire we:
1. Find the **nearest grid point** to the fire's latitude/longitude (`add_grid_cell_columns()`).
2. Look up that grid point's weather on the fire's **discovery date**.

This is simpler than finding the nearest weather station and avoids gaps in station records, but each value is an
average over a large area, so it can't capture local conditions (a windy canyon, a cool coastal valley).

In [15]:
fires = add_grid_cell_columns(fires)
print("Distinct grid cells containing fires:", fires[["cell_lat", "cell_lon"]].drop_duplicates().shape[0])
fires[["latitude", "longitude", "cell_lat", "cell_lon"]].head()

Distinct grid cells containing fires: 175


,latitude,longitude,cell_lat,cell_lon
0,40.036944,-121.005833,40.0,-121.250
1,38.933056,-120.404444,39.0,-120.625
2,38.984167,-120.735556,39.0,-120.625
3,38.559167,-119.913333,38.5,-120.000
4,38.559167,-119.933056,38.5,-120.000


`merge(..., how="left")` keeps **every fire**, even if no weather match is found (those get missing weather),
so we can count unmatched fires rather than lose them silently.

In [16]:
merged = fires.merge(
    weather,
    left_on=["cell_lat", "cell_lon", "discovery_date"],
    right_on=["cell_lat", "cell_lon", "date"],
    how="left",
).drop(columns="date")

print(f"Fires before merge: {len(fires):,}   rows after merge: {len(merged):,}")
print("Fires with no weather match:", merged["temp_max_f"].isna().sum())

Fires before merge: 251,880   rows after merge: 251,880
Fires with no weather match: 0


## Part 4: Final analysis table

In [17]:
final_columns = [
    "fire_id", "fire_name", "discovery_date", "fire_year", "fire_month", "fire_season",
    "county", "latitude", "longitude", "cell_lat", "cell_lon",
    "cause_class", "cause", "land_owner",
    "fire_size_acres", "fire_size_class", "log_fire_size", "is_large_fire",
    "temp_max_f", "relative_humidity", "wind_speed_mph", "precip_mm", "precip_prev_7day_mm",
]
analysis = merged[final_columns]
print(analysis.shape)
analysis.head()

(251880, 23)


,fire_id,fire_name,discovery_date,fire_year,fire_month,fire_season,county,latitude,longitude,cell_lat,...,land_owner,fire_size_acres,fire_size_class,log_fire_size,is_large_fire,temp_max_f,relative_humidity,wind_speed_mph,precip_mm,precip_prev_7day_mm
0,1,FOUNTAIN,2005-02-02,2005,2,Winter,Plumas,40.036944,-121.005833,40.0,...,USFS,0.10,A,0.095310,False,47.876,58.51,0.268433,0.00,41.37
1,2,PIGEON,2004-05-12,2004,5,Spring,Placer,38.933056,-120.404444,39.0,...,USFS,0.25,A,0.223144,False,67.892,49.61,0.961884,0.00,6.06
2,3,SLACK,2004-05-31,2004,5,Spring,El Dorado,38.984167,-120.735556,39.0,...,STATE OR PRIVATE,0.10,A,0.095310,False,80.312,34.96,1.252686,0.63,8.60
3,4,DEER,2004-06-28,2004,6,Summer,Alpine,38.559167,-119.913333,38.5,...,USFS,0.10,A,0.095310,False,78.548,40.19,1.073731,0.14,4.35
4,5,STEVENOT,2004-06-28,2004,6,Summer,Alpine,38.559167,-119.933056,38.5,...,USFS,0.10,A,0.095310,False,78.548,40.19,1.073731,0.14,4.35


In [18]:
analysis.isna().sum()

fire_id                0
fire_name              0
discovery_date         0
fire_year              0
fire_month             0
fire_season            0
county                 0
latitude               0
longitude              0
cell_lat               0
cell_lon               0
cause_class            0
cause                  0
land_owner             0
fire_size_acres        0
fire_size_class        0
log_fire_size          0
is_large_fire          0
temp_max_f             0
relative_humidity      0
wind_speed_mph         0
precip_mm              0
precip_prev_7day_mm    0
dtype: int64

### Save
The file is too large for GitHub, so `data/processed/` is listed in `.gitignore`.
Anyone can recreate it by running Notebooks 01–02 after downloading the raw data (see README).

In [19]:
analysis.to_csv("../data/processed/wildfires_with_weather.csv", index=False)
print("Saved", f"{len(analysis):,}", "rows")

Saved 251,880 rows


## Row count summary and limitations

| Step | Rows |
|---|---|
| California records loaded | 251,881 |
| After removing duplicate IDs | 251,881 (0 removed) |
| After removing impossible sizes / locations outside California | 251,880 (1 removed) |
| After merging with weather (left join) | 251,880 (all fires matched to weather) |

**Limitations of the matching:**
- Each weather value is an average over a ~55 km grid cell, not the exact fire location. Terrain, elevation,
  and coastal effects inside a cell are lost.
- We use weather on the **discovery date** only. How big a fire gets also depends on weather over the following
  days while it burns, which we don't include.
- Humidity and wind are **daily averages**; the afternoon low humidity and wind gusts that drive fire spread
  are stronger than these averages suggest.
- NASA POWER is modeled (reanalysis) data, not direct measurements.